In [0]:
# ---- CELL 1: config -------------------------------------------------
CATALOG = "workspace"
SCHEMA  = "cafe_karan"

BRONZE_TABLE     = f"{CATALOG}.{SCHEMA}.bronze_cafe_sales"
SILVER_TABLE     = f"{CATALOG}.{SCHEMA}.silver_cafe_sales"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.silver_cafe_sales_quarantine"

from pyspark.sql import functions as F
from itertools import chain

bronze = spark.table(BRONZE_TABLE)
print("Bronze rows:", bronze.count())   # 10000

# COMMAND ----------

# ---- CELL 2: sentinels -> real NULL ---------------------------------
# "ERROR", "UNKNOWN" and "" are not values, they are missing data wearing
# a costume. Convert them so Spark's null handling works from here on.
SENTINELS = ["ERROR", "UNKNOWN", ""]
DATA_COLS = [c for c in bronze.columns if not c.startswith("_")]

df = bronze
for c in DATA_COLS:
    df = df.withColumn(
        c,
        F.when(F.trim(F.col(c)).isin(SENTINELS), None).otherwise(F.trim(F.col(c)))
    )

display(df.limit(10))

# COMMAND ----------

# ---- CELL 3: cast to real types --------------------------------------
# Now that garbage is NULL, casting is safe: nothing silently disappears.
df = (
    df
    .withColumn("quantity",         F.col("quantity").cast("double"))
    .withColumn("price_per_unit",   F.col("price_per_unit").cast("double"))
    .withColumn("total_spent",      F.col("total_spent").cast("double"))
    .withColumn("transaction_date", F.to_date("transaction_date", "yyyy-MM-dd"))
)
df.printSchema()

# COMMAND ----------

# ---- CELL 4: remember what was missing BEFORE we repair --------------
# Provenance: a recovered value is not the same as an observed one, and
# anyone reading this table later deserves to know which is which.
df = (
    df
    .withColumn("_item_was_missing",     F.col("item").isNull())
    .withColumn("_quantity_was_missing", F.col("quantity").isNull())
    .withColumn("_price_was_missing",    F.col("price_per_unit").isNull())
    .withColumn("_total_was_missing",    F.col("total_spent").isNull())
)

# COMMAND ----------

# ---- CELL 5: price <- menu lookup by item ----------------------------
# create_map turns a Python dict into a Spark column expression you can
# look up with map[key]. coalesce = "first non-null wins".
MENU = {"Cake": 3.0, "Coffee": 2.0, "Cookie": 1.0, "Juice": 3.0,
        "Salad": 5.0, "Sandwich": 4.0, "Smoothie": 4.0, "Tea": 1.5}

menu_map = F.create_map([F.lit(x) for x in chain(*MENU.items())])

df = df.withColumn(
    "price_per_unit",
    F.coalesce(F.col("price_per_unit"), menu_map[F.col("item")])
)

# COMMAND ----------

# ---- CELL 6: item <- price, ONLY where the price is unambiguous ------
# 3.0 could be Cake or Juice; 4.0 could be Sandwich or Smoothie.
# Those stay NULL on purpose. Never invent a value to fill a gap.
UNIQUE_PRICE_TO_ITEM = {1.0: "Cookie", 1.5: "Tea", 2.0: "Coffee", 5.0: "Salad"}

price_map = F.create_map([F.lit(x) for x in chain(*UNIQUE_PRICE_TO_ITEM.items())])

df = df.withColumn(
    "item",
    F.coalesce(F.col("item"), price_map[F.col("price_per_unit")])
)

# COMMAND ----------

# ---- CELL 7: arithmetic recovery -------------------------------------
# Validated above: total == quantity * price on all 8,544 clean rows.
# Each line only fires where the target is NULL and the inputs are not,
# so nothing already known gets overwritten.
df = (
    df
    .withColumn("total_spent",
        F.coalesce(F.col("total_spent"), F.col("quantity") * F.col("price_per_unit")))
    .withColumn("quantity",
        F.coalesce(F.col("quantity"),
                   F.when(F.col("price_per_unit") != 0,
                          F.col("total_spent") / F.col("price_per_unit"))))
    .withColumn("price_per_unit",
        F.coalesce(F.col("price_per_unit"),
                   F.when(F.col("quantity") != 0,
                          F.col("total_spent") / F.col("quantity"))))
)

# COMMAND ----------

# ---- CELL 8: dedup ---------------------------------------------------
# transaction_id is the natural key. This dataset happens to have no
# duplicates, but a pipeline that only works on clean input is not a
# pipeline. Cheap insurance.
before = df.count()
df = df.dropDuplicates(["transaction_id"])
print(f"dedup: {before} -> {df.count()}")

# COMMAND ----------

# ---- CELL 9: split usable rows from unusable -------------------------
# A row without revenue cannot contribute to a single KPI. It is not
# deleted, it is quarantined -- so you can go ask someone about it.
usable     = df.filter(F.col("total_spent").isNotNull())
quarantine = df.filter(F.col("total_spent").isNull())

print("usable:     ", usable.count())      # expect 9977
print("quarantined:", quarantine.count())  # expect 23

# COMMAND ----------

# ---- CELL 10: write both ---------------------------------------------
(usable.write.format("delta").mode("overwrite")
    .option("overwriteSchema", True).saveAsTable(SILVER_TABLE))

(quarantine.write.format("delta").mode("overwrite")
    .option("overwriteSchema", True).saveAsTable(QUARANTINE_TABLE))

print("Wrote:", SILVER_TABLE, "and", QUARANTINE_TABLE)

# COMMAND ----------

# ---- CELL 11: verify the repair --------------------------------------
silver = spark.table(SILVER_TABLE)

display(silver.select(
    F.count("*").alias("rows"),
    F.sum(F.col("_total_was_missing").cast("int")).alias("total_recovered"),
    F.sum(F.col("_price_was_missing").cast("int")).alias("price_recovered"),
    F.sum(F.col("_quantity_was_missing").cast("int")).alias("qty_recovered"),
    F.sum(F.col("item").isNull().cast("int")).alias("item_still_unknown"),
))

# COMMAND ----------

# ---- CELL 12: prove the repair didn't corrupt anything ----------------
# Every row should now satisfy total == qty * price. If this is not 0,
# your recovery logic has a bug -- do NOT move on to gold.
bad = silver.filter(
    F.abs(F.col("total_spent") - F.col("quantity") * F.col("price_per_unit")) > 0.01
).count()
print("rows violating total = qty * price:", bad)   # must be 0